# ETL — Protein Structural Quality Dataset (CASP)

## Objectives

- Load the CASP protein tertiary structure dataset.
- Inspect dataset dimensions, data types, and descriptive statistics.
- Identify missing values, duplicate records, and potential data quality issues.
- Investigate duplicate observations before deciding how to handle them.
- Export a cleaned dataset for exploratory analysis and machine learning.

## Inputs

- data/raw/CASP.csv

## Outputs

- data/processed/casp_cleaned.csv
- Summary of data quality checks and cleaning decisions.

## Additional Comments

The CASP dataset contains physicochemical and structural
descriptors of predicted protein models.

The target variable is RMSD, a continuous measure of
structural deviation.

Exact duplicate records will be investigated because they
may affect model evaluation and introduce leakage if
identical records occur in both training and test sets.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [7]:

import os
from pathlib import Path

current_dir = Path.cwd()
print("Current directory:", current_dir)

Current directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [5]:
if Path.cwd().name == "jupyter_notebooks":
    os.chdir(Path.cwd().parent)

print("Working directory set to:", Path.cwd())

Working directory set to: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


Confirm the new current directory

In [6]:
print("Confirmed project directory:", Path.cwd())

Confirmed project directory: c:\Users\trasn\OneDrive\Dokumente\programing\ML fundamentals\Capstone\ProteinStruct-ML


# Section 1 — Data Loading and Initial Inspection

In [8]:
import pandas as pd
import numpy as np

# Load the original CASP dataset
df = pd.read_csv("data/raw/CASP.csv")

# Display dataset dimensions
print("Dataset shape:", df.shape)

# Display the first five observations
display(df.head())

Dataset shape: (45730, 10)


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [9]:
# Inspect dataset structure and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45730 entries, 0 to 45729
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RMSD    45730 non-null  float64
 1   F1      45730 non-null  float64
 2   F2      45730 non-null  float64
 3   F3      45730 non-null  float64
 4   F4      45730 non-null  float64
 5   F5      45730 non-null  float64
 6   F6      45730 non-null  float64
 7   F7      45730 non-null  float64
 8   F8      45730 non-null  int64  
 9   F9      45730 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 3.5 MB


---

### Initial Dataset Structure — Findings

The CASP dataset contains 45,730 observations and 10 numerical
variables, including the target variable RMSD and nine predictors
(F1–F9).

All columns contain 45,730 non-null observations, indicating
that no missing values were detected during the initial inspection.

Nine variables have the float64 data type, while F8 has the
int64 data type.

The dataset requires approximately 3.5 MB of memory and is
suitable for local analysis.

Further data quality checks will investigate duplicate records,
invalid values, and numerical distributions before cleaning.

In [10]:
# Generate descriptive statistics for all numerical variables
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RMSD,45730.0,7.748528e+00,6.118312,0.0000,2.305000,5.030000e+00,1.337900e+01,2.099900e+01
F1,45730.0,9.871597e+03,4058.138034,2392.0500,6936.680000,8.898805e+03,1.212615e+04,4.003490e+04
F2,45730.0,3.017367e+03,1464.324663,403.5000,1979.045000,2.668155e+03,3.786410e+03,1.531200e+04
F3,45730.0,3.023919e-01,0.062886,0.0925,0.258740,3.001500e-01,3.428900e-01,5.776900e-01
F4,45730.0,1.034924e+02,55.424985,10.3101,63.563900,8.774080e+01,1.336467e+02,3.693170e+02
F5,45730.0,1.368299e+06,564036.688407,319490.2166,953591.221150,1.237219e+06,1.690920e+06,5.472011e+06
F6,45730.0,1.456381e+02,69.999230,31.9704,94.757500,1.261760e+02,1.814685e+02,5.984080e+02
F7,45730.0,3.989756e+03,1993.574575,0.0000,3165.322500,3.840170e+03,4.644193e+03,1.059482e+05
F8,45730.0,6.997507e+01,56.493443,0.0000,31.000000,5.400000e+01,9.100000e+01,3.500000e+02
F9,45730.0,3.452366e+01,5.979755,15.2280,30.424725,3.529930e+01,3.887080e+01,5.530090e+01


### Descriptive Statistics — Initial Findings

The CASP dataset contains 45,730 observations, with RMSD as
the continuous target variable and nine numerical predictors.

**Target variable (RMSD):**
- Mean: 7.75 Å
- Median: 5.03 Å
- Standard deviation: 6.12 Å
- Range: 0.00–21.00 Å

The mean RMSD exceeds the median, suggesting a positively
skewed distribution. This will be investigated during EDA.

**Predictor variables:**
- The predictors exhibit substantially different numerical scales.
- F5 contains values in the millions, whereas F3 contains
  values below 1.
- F1 and F7 show large differences between their median
  and maximum values, indicating potential upper-tail outliers.
- F8 contains integer-valued observations.

**Implications for preprocessing:**

Feature scaling will be considered within the machine learning
pipelines, particularly for models sensitive to feature scale.

Potential outliers will be investigated before any removal
decisions are made. Extreme values are not necessarily errors
and may represent legitimate protein structural characteristics.

# Section 2 — Data Quality Assessment

In [11]:
# Check missing values in each column
missing_values = df.isnull().sum()

print("Missing values per column:")
display(missing_values)

# Check exact duplicate observations
duplicate_count = df.duplicated().sum()

print(f"\nNumber of exact duplicate rows: {duplicate_count}")

# Calculate percentage of duplicate rows
duplicate_percentage = (duplicate_count / len(df)) * 100

print(f"Percentage of duplicate rows: {duplicate_percentage:.2f}%")

Missing values per column:


RMSD    0
F1      0
F2      0
F3      0
F4      0
F5      0
F6      0
F7      0
F8      0
F9      0
dtype: int64


Number of exact duplicate rows: 1711
Percentage of duplicate rows: 3.74%


### Investigation of Duplicate Observations

In [12]:
# Display all rows belonging to duplicate groups
duplicate_rows = df[df.duplicated(keep=False)]

print("Rows belonging to duplicate groups:", len(duplicate_rows))

# Show a sample of repeated observations
display(duplicate_rows.head(10))

# Count how often each complete observation occurs
duplicate_frequency = (
    df.value_counts()
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)

display(duplicate_frequency.head(10))

Rows belonging to duplicate groups: 3162


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
21,18.404,13605.50,4766.37,0.35032,139.7930,1.925985e+06,182.1950,4672.34,112,26.4540
30,16.897,11155.60,3383.81,0.30332,142.7030,1.573166e+06,159.0590,4251.55,133,29.6432
63,18.481,13117.30,6444.76,0.49131,135.2140,1.794766e+06,231.0990,5094.41,140,28.2733
74,3.092,9763.57,2749.97,0.28165,111.4310,1.357573e+06,171.1030,4202.52,52,33.7255
79,10.252,7182.05,2359.80,0.32856,55.8177,9.733760e+05,101.3970,2801.82,19,40.9580
83,4.430,13179.00,3998.54,0.30340,149.9450,1.839480e+06,218.1110,4690.99,111,29.1008
85,19.190,16982.20,4743.63,0.27932,183.5550,2.347319e+06,197.3280,5738.57,116,25.2664
93,5.280,6387.19,1555.71,0.24356,58.4257,8.409970e+05,76.1158,3115.79,34,39.6168
96,10.029,6063.73,2388.49,0.39389,59.2914,8.577841e+05,85.3898,2693.96,35,40.3893
101,5.233,8569.51,2095.52,0.24453,85.6867,1.215084e+06,118.4690,3889.37,73,34.4689


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9,frequency
0,7.190,13475.40,4814.93,0.35731,168.5500,1.877844e+06,227.6050,4644.75,99,29.7563,17
1,4.070,5811.82,1053.23,0.18122,52.5591,7.999772e+05,65.2332,3034.98,24,39.7659,9
2,1.730,4000.26,1087.13,0.27176,33.7320,5.694944e+05,46.0390,1773.46,12,44.4892,8
3,1.697,14170.50,3520.75,0.24845,186.4070,1.937926e+06,214.6660,4581.39,179,28.9962,7
4,2.055,4074.87,708.80,0.17394,39.8429,5.371195e+05,60.1525,2562.83,11,43.4461,6
5,14.744,8646.41,2574.14,0.29771,52.0932,1.173612e+06,114.6390,3889.60,26,36.0445,5
6,7.301,16788.70,5483.99,0.32664,184.3400,2.335620e+06,279.7950,5604.02,98,23.6274,5
7,1.720,8630.52,2458.32,0.28484,95.4877,1.187140e+06,151.2650,3225.39,10,36.7779,5
8,4.633,5942.66,1448.36,0.24372,63.6140,8.259933e+05,80.9010,2102.95,61,42.0209,5
9,11.733,4236.27,1376.40,0.32490,35.0902,5.765423e+05,54.2159,1426.57,34,44.3511,4


---

In [13]:
# Identify predictor columns
feature_columns = [col for col in df.columns if col != "RMSD"]

# Count distinct RMSD values for each combination of predictors
target_variation = (
    df.groupby(feature_columns)["RMSD"]
    .nunique()
)

# Count predictor combinations associated with multiple RMSD values
conflicting_groups = (target_variation > 1).sum()

print(
    "Predictor combinations with different RMSD values:",
    conflicting_groups
)

Predictor combinations with different RMSD values: 38


### Duplicate Records — Initial Findings

The dataset contains 1,711 exact duplicate records,
representing 3.74% of the original 45,730 observations.

This leaves 44,019 unique complete observations.

Exact duplicates may influence model training and evaluation,
particularly if identical observations are distributed across
training and test datasets.

Further investigation will examine duplicate frequencies and
whether identical predictor combinations are associated with
different RMSD values.

No records have been removed at this stage.

### Handling Duplicate Observations

In [14]:
# Preserve the original dataset
df_original = df.copy()

# Remove only exact duplicates across all columns
df_clean = df.drop_duplicates().reset_index(drop=True)

# Compare dataset dimensions
print("Original dataset:", df_original.shape)
print("Cleaned dataset:", df_clean.shape)

# Verify exact duplicates were removed
print("Remaining exact duplicates:", df_clean.duplicated().sum())

# Check whether predictor combinations with differing RMSD remain
feature_columns = [col for col in df_clean.columns if col != "RMSD"]

conflicting_groups_clean = (
    df_clean.groupby(feature_columns)["RMSD"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "Predictor combinations with different RMSD values:",
    conflicting_groups_clean
)

Original dataset: (45730, 10)
Cleaned dataset: (44019, 10)
Remaining exact duplicates: 0
Predictor combinations with different RMSD values: 38


### Data Cleaning Decisions

The original dataset contained 1,711 exact duplicate records
(3.74% of observations).

These records were removed to reduce redundant observations
and the risk of duplicate-related bias in model evaluation.

After deduplication, 44,019 unique complete records remained.

Further investigation identified 38 combinations of predictor
values (F1–F9) associated with more than one RMSD value.

These observations were retained because differing target values
do not necessarily indicate data errors. They may reflect
information not represented by the available descriptors.

No outlier removal or feature scaling was performed during ETL.

To avoid identical predictor combinations crossing training
and test boundaries, group-aware splitting will be considered
during the machine learning stage.

In [15]:
from pathlib import Path

# Create output directory if necessary
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Export cleaned dataset
output_path = output_dir / "casp_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: data\processed\casp_cleaned.csv


## Section 3 — Data Validation

In [16]:
# Reload the processed dataset
df_validated = pd.read_csv("data/processed/casp_cleaned.csv")

# Check dimensions
print("Dataset shape:", df_validated.shape)

# Check missing values
print("Total missing values:", df_validated.isnull().sum().sum())

# Check exact duplicates
print("Exact duplicate rows:", df_validated.duplicated().sum())

# Check column names
print("Columns:", df_validated.columns.tolist())

# Display first five rows
display(df_validated.head())

Dataset shape: (44019, 10)
Total missing values: 0
Exact duplicate rows: 0
Columns: ['RMSD', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'F9']


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [17]:
# Verify that the exported dataset meets our expectations
assert df_validated.shape == (44019, 10), "Unexpected dataset dimensions"
assert df_validated.isnull().sum().sum() == 0, "Missing values detected"
assert df_validated.duplicated().sum() == 0, "Duplicates detected"
assert df_validated.columns.tolist() == df.columns.tolist(), "Column mismatch"

print("All data validation checks passed!")

All data validation checks passed!


### ETL Validation Results

The processed CASP dataset was reloaded successfully.

Validation checks confirmed:
- 44,019 observations and 10 columns
- No missing values
- No exact duplicate observations
- All expected columns preserved

The cleaned dataset is stored at:
`data/processed/casp_cleaned.csv`

The original dataset remains unchanged in:
`data/raw/CASP.csv`

The processed dataset is ready for exploratory data analysis
and subsequent machine learning workflows.

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
